# 03 — Train DeepScalper
Trains one `DuelingQNetwork` per ticker using **Double DQN + Prioritized Experience Replay**.

**Training loop per ticker:**
- Up to `MAX_EPISODES=200` episodes (each episode = 1 random training day).
- Early stopping if validation Sharpe does not improve for `PATIENCE=20` episodes.
- Best model (by val Sharpe) is saved as `{TICKER}.pth`.

**Input:**  `/content/drive/MyDrive/algo_trader/data/features/{TICKER}_train.npz`  
**Output:** `/content/drive/MyDrive/algo_trader/weights/{TICKER}.pth`

> Enable GPU runtime: **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!pip install -q torch torchvision gymnasium numpy pandas pytz tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
FEAT_DIR    = '/content/drive/MyDrive/algo_trader/data/features'
WEIGHTS_DIR = '/content/drive/MyDrive/algo_trader/weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print(f'Weights will be saved to: {WEIGHTS_DIR}')

In [ ]:
REPO_URL = 'https://github.com/YOUR_GITHUB_USERNAME/deepscalper_copilot.git'
REPO_DIR = '/content/deepscalper_copilot'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

if REPO_DIR + '/algo_trader' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/algo_trader')
print('Repo on path ✓')

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Hyperparameters — keep in sync with config.py
LOOKBACK_BARS = 60
INPUT_DIM     = 11
ACTION_DIM    = 3
HIDDEN_SIZE   = 128
FC_SIZE       = 256
DROPOUT_RATE  = 0.2

MAX_EPISODES  = 200
PATIENCE      = 20     # Early stopping patience (episodes)
EVAL_EVERY    = 5      # Validate every N training episodes
EVAL_EPISODES = 10     # Episodes to run for each validation Sharpe estimate

SP100_TICKERS = [
    'AAPL','MSFT','AMZN','NVDA','GOOGL','GOOG','META','TSLA','BRK.B','UNH',
    'LLY','JPM','V','AVGO','XOM','MA','COST','PG','JNJ','HD',
    'ABBV','ORCL','BAC','WMT','NFLX','KO','CRM','CVX','MRK','AMD',
    'CSCO','PEP','ACN','LIN','TMO','MCD','ABT','IBM','GE','TXN',
    'PM','GS','ISRG','CAT','AXP','SPGI','AMGN','RTX','PFE','BKNG',
    'DHR','MS','INTU','BLK','T','VRTX','HON','NEE','UNP','SYK',
    'C','LOW','TJX','ADP','GILD','DE','PANW','BMY','AMAT','MDT',
    'PLD','SBUX','ADI','TMUS','ETN','SCHW','CB','MMC','BA','SO',
    'MO','WFC','UPS','CI','MDLZ','DUK','CL','INTC','REGN','PH',
    'EOG','SLB','ELV','APD','MCK','COF','ZTS','BSX','GEV','CME',
]

In [ ]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from colab.deepscalper.agent import DeepScalperAgent
from colab.deepscalper.environment import TradingEnv
from colab.deepscalper.utils import compute_sharpe


def evaluate_agent(agent: DeepScalperAgent, env: TradingEnv, n_episodes: int) -> float:
    """Run n_episodes in eval mode and return the annualised Sharpe ratio."""
    episode_rewards = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        total_r = 0.0
        while not done:
            action = agent.select_action(obs, eval_mode=True)
            obs, r, terminated, truncated, _ = env.step(action)
            total_r += r
            done = terminated or truncated
        episode_rewards.append(total_r)
    return compute_sharpe(episode_rewards)


training_log = []

for ticker in tqdm(SP100_TICKERS, desc='Tickers'):
    weights_path = f'{WEIGHTS_DIR}/{ticker}.pth'
    if os.path.exists(weights_path):
        print(f'{ticker}: weights already exist — skipping.')
        continue

    train_path = f'{FEAT_DIR}/{ticker}_train.npz'
    val_path   = f'{FEAT_DIR}/{ticker}_val.npz'
    if not os.path.exists(train_path):
        print(f'WARNING: {train_path} missing — skipping {ticker}.')
        continue

    # Load datasets
    train_data      = np.load(train_path)
    X_train         = train_data['X']           # (n_windows, 60, 11)
    train_day_starts = train_data['day_starts'].tolist()

    val_data        = np.load(val_path)
    X_val           = val_data['X']
    val_day_starts  = val_data['day_starts'].tolist()

    if len(train_day_starts) < 5:
        print(f'WARNING: {ticker} has only {len(train_day_starts)} training days — skipping.')
        continue

    # Environments
    # TradingEnv expects raw features (n_bars, 11) — we pass the window matrix
    # Each window is already a (60, 11) slice; use windows as bar features
    # (day_starts still indexes into windows correctly since window[i] ends at bar i+60)
    train_env = TradingEnv(
        features=X_train.reshape(-1, INPUT_DIM),    # Flatten back to (n_bars, 11)
        day_starts=train_day_starts,
        lookback_bars=LOOKBACK_BARS,
    )
    val_env = TradingEnv(
        features=X_val.reshape(-1, INPUT_DIM),
        day_starts=val_day_starts,
        lookback_bars=LOOKBACK_BARS,
    )

    # Instantiate agent
    agent = DeepScalperAgent(
        lookback_bars=LOOKBACK_BARS,
        input_dim=INPUT_DIM,
        action_dim=ACTION_DIM,
        hidden_size=HIDDEN_SIZE,
        fc_size=FC_SIZE,
        dropout_rate=DROPOUT_RATE,
        device=DEVICE,
    )

    best_sharpe    = -np.inf
    patience_count = 0

    for episode in range(1, MAX_EPISODES + 1):
        # Training episode
        obs, _ = train_env.reset()
        done = False
        while not done:
            action    = agent.select_action(obs)
            next_obs, reward, terminated, truncated, _ = train_env.step(action)
            done = terminated or truncated
            agent.store(obs, action, reward, next_obs, done)
            agent.learn()
            obs = next_obs

        # Periodic validation
        if episode % EVAL_EVERY == 0:
            val_sharpe = evaluate_agent(agent, val_env, EVAL_EPISODES)

            if val_sharpe > best_sharpe:
                best_sharpe    = val_sharpe
                patience_count = 0
                agent.save(weights_path)
            else:
                patience_count += 1

            if patience_count >= PATIENCE:
                print(f'{ticker}: early stop at episode {episode} (best Sharpe={best_sharpe:.3f})')
                break

    # Ensure weights were saved (even if agent never improved during eval)
    if not os.path.exists(weights_path):
        agent.save(weights_path)

    training_log.append({'ticker': ticker, 'best_val_sharpe': round(best_sharpe, 4)})
    print(f'{ticker}: best val Sharpe = {best_sharpe:.4f}')


print('\n=== TRAINING COMPLETE ===')
if training_log:
    df_log = pd.DataFrame(training_log).sort_values('best_val_sharpe', ascending=False)
    print(df_log.to_string(index=False))

# Save training log to Drive for reference
log_path = '/content/drive/MyDrive/algo_trader/training_log.csv'
pd.DataFrame(training_log).to_csv(log_path, index=False)
print(f'\nTraining log saved → {log_path}')